# Cotton-PAMT: Multimodal Transformer for Rapid Cry Gene Detection in Bt Cotton
### Adapted from PAMT (Pathway-Aware Multimodal Transformer) for Dip-PCR & Spectral Analysis

**Project Title:** Rapid Detection of Cry Genes in Bt Cotton Using Dip-Based PCR and AI-Assisted Spectral Analysis  
**Collaborators:** Mahindra University, ICAR-IISR Indore, APS Labs  
**Target Diagnostic Classes:**  
- `0`: **Non-Bt Cotton** (Conventional / Refuge - Negative for Cry genes)  
- `1`: **Cry1Ac Positive** (Bollgard-I, ~300 bp amplicon, ~530 nm spectral shift)  
- `2`: **Cry2Ab Positive** (~750 bp amplicon, ~585 nm spectral shift)  
- `3`: **Stacked Cry1Ac + Cry2Ab Positive** (Bollgard-II, dual amplicons ~300 bp & ~750 bp)  

---

## Step 1: Mount Google Drive & Environment Setup
This cell mounts your Google Drive so that all trained model weights, evaluation plots, and mobile ONNX files are **permanently saved directly to your Google Drive** (`My Drive/Cotton_PAMT_Project`).

In [ ]:
# Install dependencies for training, visualization and ONNX export
!pip install -q matplotlib seaborn scikit-learn onnx onnxscript

import os
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# --- Google Drive Mounting ---
try:
    from google.colab import drive
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
    OUTPUT_DIR = "/content/drive/MyDrive/Cotton_PAMT_Project"
except Exception as e:
    print("Running outside Google Colab or Drive mount skipped. Saving to local output folder.")
    OUTPUT_DIR = "./cotton_pamt_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\n [OUTPUT FOLDER] All outputs will be saved to:\n--> {OUTPUT_DIR}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nExecution Device: {device}")
if device.type == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

## Step 2: Realistic Synthetic Dataset Generator
Simulates paired Dip-PCR Gel band profiles (spatial bands: 300 bp, 750 bp, Sad1 plant reference), continuous UV-Vis colorimetric absorbance curves (400–700 nm), and molecular genetic target markers (Cry1Ac, Cry2Ab, 35S promoter, NOS terminator, Sad1).

In [ ]:
def generate_synthetic_bt_cotton_dataset(num_samples=1000, seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    samples = []
    classes = [0, 1, 2, 3]
    wavelengths = np.linspace(400, 700, 60)
    
    for i in range(num_samples):
        label = np.random.choice(classes, p=[0.25, 0.25, 0.25, 0.25])
        
        # 1. Dip-PCR Gel Lane Image Feature [32 height bins, 8 filter channels]
        lane = np.random.normal(0.05, 0.02, (32, 8)).astype(np.float32)
        # Plant internal control (Sad1/GhActin) present in all valid samples (bins 8-9)
        lane[8:10, :] += np.random.uniform(0.6, 0.9, (2, 8))
        
        # 2. UV-Vis / Colorimetric Dipstick Absorbance Spectrum (400 - 700 nm, 60 bands)
        spectrum = (0.15 + 0.1 * np.exp(-(wavelengths - 420)**2 / (2 * 50**2))).astype(np.float32)
        spectrum += np.random.normal(0, 0.01, 60).astype(np.float32)
        
        # 3. Target Molecular Marker Incidences [Cry1Ac, Cry2Ab, 35S, NOS, Sad1]
        markers = np.array([0.0, 0.0, 0.0, 0.0, 1.0], dtype=np.float32) # Sad1 is always 1.0
        
        if label == 1: # Cry1Ac
            lane[11:13, :] += np.random.uniform(0.7, 1.0, (2, 8)) # ~300 bp amplicon band
            spectrum += (0.5 * np.exp(-(wavelengths - 530)**2 / (2 * 15**2))).astype(np.float32) # 530 nm peak
            markers[0] = 1.0
            markers[2] = 1.0
            markers[3] = 1.0
        elif label == 2: # Cry2Ab
            lane[21:23, :] += np.random.uniform(0.7, 1.0, (2, 8)) # ~750 bp amplicon band
            spectrum += (0.45 * np.exp(-(wavelengths - 585)**2 / (2 * 18**2))).astype(np.float32) # 585 nm peak
            markers[1] = 1.0
            markers[2] = 1.0
            markers[3] = 1.0
        elif label == 3: # Stacked (Cry1Ac + Cry2Ab)
            lane[11:13, :] += np.random.uniform(0.65, 0.95, (2, 8))
            lane[21:23, :] += np.random.uniform(0.65, 0.95, (2, 8))
            spectrum += (0.45 * np.exp(-(wavelengths - 530)**2 / (2 * 15**2))).astype(np.float32)
            spectrum += (0.40 * np.exp(-(wavelengths - 585)**2 / (2 * 18**2))).astype(np.float32)
            markers[0] = 1.0
            markers[1] = 1.0
            markers[2] = 1.0
            markers[3] = 1.0
            
        lane = np.clip(lane, 0, 1.5).astype(np.float32)
        spectrum = np.clip(spectrum, 0, 2.0).astype(np.float32)
        
        samples.append({
            "lane": lane,
            "spectrum": spectrum,
            "markers": markers,
            "label": label
        })
    return samples

class CottonDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return (
            torch.tensor(item["lane"], dtype=torch.float32),
            torch.tensor(item["spectrum"], dtype=torch.float32),
            torch.tensor(item["markers"], dtype=torch.float32),
            torch.tensor(item["label"], dtype=torch.long)
        )

print(" [SUCCESS] Dataset generator compiled cleanly!")

## Step 3: Model Architecture (Cotton-PAMT)
Implements:
1. **GelImageEncoder:** Spatial patch transformer across gel electrophoresis migration bands.
2. **SpectralMarkerEncoder:** Transformer over continuous UV-Vis spectra and genetic markers.
3. **Cross-Attention Fusion:** Spectral queries attend to visual band tokens with clean residual normalization.
4. **Attention Extraction:** Allows visualization of attention weights for interpretability.

In [ ]:
class SpectralMarkerEncoder(nn.Module):
    def __init__(self, spec_dim=60, marker_dim=5, embed_dim=64, num_heads=4):
        super().__init__()
        self.spec_proj = nn.Sequential(
            nn.Linear(spec_dim, 128),
            nn.GELU(),
            nn.Linear(128, embed_dim)
        )
        self.marker_embed = nn.Linear(1, embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, marker_dim + 1, embed_dim))
        
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=0.1,
            activation="gelu",
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=2)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, spectrum, markers):
        spec_token = self.spec_proj(spectrum).unsqueeze(1)
        marker_tokens = self.marker_embed(markers.unsqueeze(-1))
        tokens = torch.cat([spec_token, marker_tokens], dim=1)
        tokens = tokens + self.pos_embed
        return self.norm(self.transformer(tokens))

class GelImageEncoder(nn.Module):
    def __init__(self, num_patches=32, in_dim=8, embed_dim=64, num_heads=4):
        super().__init__()
        self.patch_proj = nn.Linear(in_dim, embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=0.1,
            activation="gelu",
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=3)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        tokens = self.patch_proj(x) + self.pos_embed
        return self.norm(self.transformer(tokens))

class CottonPAMT(nn.Module):
    def __init__(self, num_classes=4, embed_dim=64):
        super().__init__()
        self.gel_encoder = GelImageEncoder(num_patches=32, in_dim=8, embed_dim=embed_dim)
        self.spec_encoder = SpectralMarkerEncoder(spec_dim=60, marker_dim=5, embed_dim=embed_dim)
        
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=4,
            dropout=0.1,
            batch_first=True
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Linear(embed_dim * 2, embed_dim)
        )
        
        # Diagnostic classification head
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, gel_img, spectrum, markers, return_attn=False):
        gel_feats = self.gel_encoder(gel_img)
        spec_feats = self.spec_encoder(spectrum, markers)
        
        attn_out, attn_weights = self.cross_attn(query=spec_feats, key=gel_feats, value=gel_feats)
        
        # Clean single-pass residual normalization
        x_norm = self.norm1(spec_feats + attn_out)
        fused = self.norm2(x_norm + self.mlp(x_norm))
        
        concat = torch.cat([spec_feats.mean(dim=1), fused.mean(dim=1)], dim=-1)
        logits = self.classifier(concat)
        
        if return_attn:
            return logits, attn_weights
        return logits

print(" [SUCCESS] Cotton-PAMT model architecture compiled cleanly!")

## Step 4: Model Training with Loss & Accuracy Tracking
Trains for 15 epochs on GPU and saves the best model weights directly to your Google Drive.

In [ ]:
# Prepare 800 train and 200 hold-out test samples
raw_data = generate_synthetic_bt_cotton_dataset(num_samples=1000, seed=42)
train_loader = DataLoader(CottonDataset(raw_data[:800]), batch_size=32, shuffle=True)
test_loader = DataLoader(CottonDataset(raw_data[800:]), batch_size=32, shuffle=False)

model = CottonPAMT(num_classes=4, embed_dim=64).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,} ({total_params/1e6:.3f} M)")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

epochs = 15
history = {"loss": [], "train_acc": []}

print(f"\nTraining Cotton-PAMT on {device}...")
start_t = time.perf_counter()

for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = 0.0
    correct = 0
    total = 0
    
    for gel, spec, mark, y in train_loader:
        gel, spec, mark, y = gel.to(device), spec.to(device), mark.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(gel, spec, mark)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * y.size(0)
        preds = out.argmax(dim=-1)
        correct += (preds == y).sum().item()
        total += y.size(0)
        
    train_loss = epoch_loss / total
    train_acc = (correct / total) * 100
    history["loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    
    if epoch % 3 == 0 or epoch == epochs:
        print(f"Epoch [{epoch:02d}/{epochs}] - Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")

elapsed = time.perf_counter() - start_t
print(f"\nTraining finished in {elapsed:.2f} seconds!")

# Save trained PyTorch model weights directly to Google Drive
weights_path = os.path.join(OUTPUT_DIR, "cotton_pamt_best.pth")
torch.save(model.state_dict(), weights_path)
print(f"\n [SAVED TO DRIVE] Best weights saved to: {weights_path}")

## Step 5: Test Set Evaluation & Save Plots to Google Drive

In [ ]:
model.eval()
all_preds = []
all_targets = []
class_names = ["Non-Bt", "Cry1Ac", "Cry2Ab", "Stacked"]

with torch.no_grad():
    for gel, spec, mark, y in test_loader:
        gel, spec, mark, y = gel.to(device), spec.to(device), mark.to(device), y.to(device)
        preds = model(gel, spec, mark).argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y.cpu().numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
test_accuracy = (all_preds == all_targets).mean() * 100

print(f"Overall Test Accuracy: {test_accuracy:.2f}%")
rep = classification_report(all_targets, all_preds, target_names=class_names)
print("\n--- Detailed Classification Report ---")
print(rep)

# Save classification report text to Drive
report_path = os.path.join(OUTPUT_DIR, "classification_report.txt")
with open(report_path, "w") as f:
    f.write(f"Overall Test Accuracy: {test_accuracy:.2f}%\n\n" + rep)

# Plot Loss Curve & Confusion Matrix and Save Figure to Drive
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, epochs + 1), history["loss"], marker="o", label="Loss", color="tab:red")
plt.title("Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Cross Entropy Loss")
plt.grid(True)

plt.subplot(1, 2, 2)
cm = confusion_matrix(all_targets, all_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title(f"Test Confusion Matrix (Acc: {test_accuracy:.1f}%)")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, "training_and_confusion_matrix.png")
plt.savefig(fig_path, dpi=300)
plt.show()
print(f" [SAVED TO DRIVE] Evaluation plot saved to: {fig_path}")

## Step 6: Attention Interpretability Heatmap (Saved to Drive)
Visualizes which Dip-PCR gel migration bands the spectral queries attend to for diagnostic verification.

In [ ]:
sample_gel, sample_spec, sample_mark, sample_y = test_loader.dataset[0]
sample_gel = sample_gel.unsqueeze(0).to(device)
sample_spec = sample_spec.unsqueeze(0).to(device)
sample_mark = sample_mark.unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    _, attn_map = model(sample_gel, sample_spec, sample_mark, return_attn=True)
    if attn_map.ndim == 4:
        attn_weights = attn_map[0].mean(dim=0).detach().cpu().numpy()
    else:
        attn_weights = attn_map[0].detach().cpu().numpy()

plt.figure(figsize=(10, 4))
query_labels = ["UV-Vis", "Cry1Ac", "Cry2Ab", "35S", "NOS", "Sad1"]
sns.heatmap(attn_weights, cmap="viridis", yticklabels=query_labels)
plt.title(f"Cross-Attention Map (True Label: {class_names[sample_y.item()]})")
plt.xlabel("Gel Lane Spatial Migration Bins (100 bp to 1500 bp)")
plt.ylabel("Spectral & Marker Queries")
plt.tight_layout()

attn_fig_path = os.path.join(OUTPUT_DIR, "cross_attention_interpretability_map.png")
plt.savefig(attn_fig_path, dpi=300)
plt.show()
print(f" [SAVED TO DRIVE] Attention map saved to: {attn_fig_path}")

## Step 7: Export Model to ONNX & TorchScript for Mobile Deployment (Phase IV)
Produces an ultra-lightweight (~1 MB) model saved directly to your Google Drive ready for offline mobile inference in Android / Flutter.

In [ ]:
model.eval().to("cpu")
dummy_gel = torch.randn(1, 32, 8)
dummy_spec = torch.randn(1, 60)
dummy_mark = torch.randn(1, 5)

# 1. Export to TorchScript
ts_path = os.path.join(OUTPUT_DIR, "cotton_pamt_mobile.pt")
try:
    scripted_model = torch.jit.trace(model, (dummy_gel, dummy_spec, dummy_mark), check_trace=False)
    scripted_model.save(ts_path)
    print(f" [SAVED TO DRIVE] Exported TorchScript model to: {ts_path}")
except Exception as e:
    print("TorchScript export notice:", e)

# 2. Export to ONNX
onnx_path = os.path.join(OUTPUT_DIR, "cotton_pamt.onnx")
try:
    torch.onnx.export(
        model,
        (dummy_gel, dummy_spec, dummy_mark),
        onnx_path,
        input_names=["gel_lane", "spectrum", "markers"],
        output_names=["diagnostic_logits"],
        dynamic_axes={
            "gel_lane": {0: "batch_size"},
            "spectrum": {0: "batch_size"},
            "markers": {0: "batch_size"},
            "diagnostic_logits": {0: "batch_size"}
        },
        opset_version=14
    )
    print(f" [SAVED TO DRIVE] Exported ONNX model to: {onnx_path}")
except Exception as e:
    print("ONNX export notice:", e)

print(f"\n=== All Artifacts Successfully Saved to Google Drive ({OUTPUT_DIR}) ===")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / (1024 * 1024)
    print(f"- {f:<35} ({size:.2f} MB)")

## Step 8: Direct ZIP Download to Your Laptop (Backup)
If you also want an instant copy downloaded directly into your laptop's **Downloads** folder, run this cell:

In [ ]:
from google.colab import files
!zip -r cotton_pamt_all_outputs.zip "{OUTPUT_DIR}"
files.download("cotton_pamt_all_outputs.zip")
print(" Initiated direct download of cotton_pamt_all_outputs.zip to your computer!")